# CM1(2) - Excel coursework

In [ ]:
1+1

2

In [ ]:
32802 - 293/2

32655.5

## Question 1

### i and ii: Extending the Life Table

In [ ]:
# Load the mortality data 
df = pd.read_csv('AM92csv.csv', skiprows=4, names=['Age', 'q_[x]', 'q_[x-1]+1', 'q_x'])
df = df.dropna(subset=['Age']).reset_index(drop=True)
df['Age'] = df['Age'].astype(int)
idx = {int(age): i for i, age in enumerate(df['Age'])}

# convert ALL q columns to numeric
df['q_x']       = pd.to_numeric(df['q_x'],       errors='coerce')
df['q_[x]']     = pd.to_numeric(df['q_[x]'],     errors='coerce')
df['q_[x-1]+1'] = pd.to_numeric(df['q_[x-1]+1'], errors='coerce')

# --- Build ultimate l_x ---
radix = 10000.0
df['l_x'] = 0.0
df.at[0, 'l_x'] = radix

for i in range(1, len(df)):
    l_prev = np.float64(df['l_x'].iloc[i-1])
    q_prev = np.float64(df['q_x'].iloc[i-1])
    df.at[i, 'l_x'] = l_prev * (1.0 - q_prev)

for i in range(len(df)):
    age = int(df['Age'].iloc[i])
    q1  = df['q_[x-1]+1'].iloc[i]
    if age <= 91 and pd.notna(q1) and (i + 1 < len(df)):
        df.at[i, 'l_[x-1]+1'] = np.float64(df['l_x'].iloc[i+1]) / (1.0 - np.float64(q1))

for i in range(len(df)):
    age = int(df['Age'].iloc[i])
    q0  = df['q_[x]'].iloc[i]
    if age <= 90 and pd.notna(q0) and (i + 1 < len(df)):
        df.at[i, 'l_[x]'] = np.float64(df['l_[x-1]+1'].iloc[i+1]) / (1.0 - np.float64(q0))

# --- Verify 10p[50] and 10p50 ---
# .index lookup instead of hardcoded indices 
idx_50 = df.index[df['Age'] == 50][0]
idx_60 = df.index[df['Age'] == 60][0]

l_x_60      = np.float64(df['l_x'].iloc[idx_60])
l_select_50 = np.float64(df['l_[x]'].iloc[idx_50])
l_ult_50    = np.float64(df['l_x'].iloc[idx_50])

p10_select_50 = l_x_60 / l_select_50
p10_ult_50    = l_x_60 / l_ult_50

print(f"10p[50] = {p10_select_50:.6f}")
print(f"10p50   = {p10_ult_50:.6f}")

10p[50] = 0.956843
10p50   = 0.956255
